# RAG Pipeline — PDF Document Assistant

This notebook builds a Retrieval-Augmented Generation (RAG) pipeline that loads PDF documents, chunks and embeds them, stores the embeddings in a FAISS vector database, and uses GPT to answer questions grounded in the document content.

## Install Dependencies

In [1]:
%pip install langchain langchain-openai langchain-community langchain-text-splitters langchain-classic langchain-core faiss-cpu pypdf

Note: you may need to restart the kernel to use updated packages.


## Imports

In [2]:
import os
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_classic.chains import RetrievalQA
from langchain_core.prompts import PromptTemplate

C:\Users\rohit\AppData\Local\Temp\ipykernel_24720\1594486673.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


## Step 1: Load PDF Documents

`PyPDFLoader` reads each page of a PDF as a separate `Document` object, preserving metadata such as the source filename and page number.

In [3]:
pdf_files = [
    "data/1762517560_course_end_project_problem_statement.pdf",
    "data/ref-FAISS.pdf",
    "data/ref-PyPDFLoader.pdf"
]

documents = []
for file in pdf_files:
    loader = PyPDFLoader(file_path=file, extraction_mode="layout")
    documents.extend(loader.load())

print(f"Loaded {len(documents)} pages across {len(pdf_files)} PDFs")

Loaded 11 pages across 3 PDFs


## Step 2: Chunk Documents

`RecursiveCharacterTextSplitter` splits text using multiple separators in order (paragraphs → lines → words) to keep chunks within the size limit without cutting mid-sentence. A 50-character overlap between chunks preserves context at boundaries.

In [4]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
docs = text_splitter.split_documents(documents)

avg_size = sum(len(d.page_content) for d in docs) / len(docs)
print(f"Total chunks: {len(docs)}")
print(f"Average chunk size: {avg_size:.0f} chars")

Total chunks: 51
Average chunk size: 353 chars


## Step 3: Configure OpenAI API Key

In [ ]:
os.environ["OPENAI_API_KEY"] = "your-api-key-here"

## Step 4: Generate Embeddings and Store in FAISS

`text-embedding-ada-002` converts each chunk into a 1536-dimensional vector representing its semantic meaning. These vectors are stored in a FAISS `IndexFlatL2` index, which supports fast nearest-neighbour search using Euclidean distance.

In [6]:
embeddings = OpenAIEmbeddings(
    model="text-embedding-ada-002",
    api_key=os.environ["OPENAI_API_KEY"]
)
vectorstore = FAISS.from_documents(docs, embeddings)

print("Embeddings generated and stored in FAISS vectorstore")

Embeddings generated and stored in FAISS vectorstore


## Step 5: Initialise the LLM and Custom Prompt

A custom prompt template constrains the LLM to answer using only the retrieved context, reducing hallucinations.

In [10]:
llm = ChatOpenAI(
    model="gpt-5-mini",
    api_key=os.environ["OPENAI_API_KEY"]
)

prompt = PromptTemplate(
    input_variables=["context", "question"],
    template="Answer in 2-3 sentences using only the context below.\n\nContext: {context}\n\nQuestion: {question}\nAnswer:"
)

## Step 6: Build the RetrievalQA Chain

The chain retrieves the most relevant chunks from FAISS for a given query and passes them to the LLM to generate a grounded answer.

In [11]:
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=vectorstore.as_retriever(),
    return_source_documents=True,
    chain_type_kwargs={"prompt": prompt}
)

## Step 7: Run a Sample Query

Submit a question to the pipeline and display the generated answer along with the source document chunks that were retrieved.

In [12]:
query = "Should I use FAISS and is that part of the requirements?"

result = qa_chain.invoke(query)
answer = result['result']
source_docs = result['source_documents']

print(f"Query: {query}")
print("\nAnswer:")
print(answer)
print("\nSource Documents:")
for i, doc in enumerate(source_docs, 1):
    content = doc.page_content
    source = doc.metadata.get('source', 'unknown')
    page = doc.metadata.get('page', 'unknown')
    truncated = '(truncated)' if len(content) > 50 else ''
    print(f"\nSource {i} [{source}, page {page}]:-")
    print(f"{content[:50]}{truncated}")

Query: Should I use FAISS and is that part of the requirements?

Answer:
Yes — FAISS is explicitly listed among the required tools (Visual Studio Code, Python, FAISS, LangChain, and OpenAI API), so you should use it for the project. Install it using the recommended conda packages (conda install -c pytorch faiss-cpu or conda install -c pytorch faiss-gpu) and do not install both.

Source Documents:

Source 1 [data/ref-FAISS.pdf, page 0]:-
Faiss


Faiss is a library for eﬃcient similarity (truncated)

Source 2 [data/1762517560_course_end_project_problem_statement.pdf, page 0]:-
Instructions

    •   Tools: Use Visual Studio Cod(truncated)

Source 3 [data/ref-FAISS.pdf, page 1]:-
Install


The recommended way to install Faiss is (truncated)

Source 4 [data/ref-FAISS.pdf, page 3]:-
Image credit: Yusuke Matsui, thanks for allowing u(truncated)
